# 2FA Authenticator — Neon Database First-Time Setup

This notebook sets up the **Neon Postgres** database for the 2FA Authenticator platform in 4 steps:

1. Install the database driver
2. Connect to your Neon database (paste your connection string when asked)
3. Create the tables the app expects
4. Generate the two secret keys you'll paste into Vercel

**How to use:** Runtime → Run all (or run cells top to bottom). Takes about a minute.

> **Before you start** — create a free project at [neon.tech](https://console.neon.tech), open it, click **Connect**, and copy the **pooled** connection string (the one with **`-pooler`** in the host and `?sslmode=require` at the end).

## Step 1 — Install the driver

In [ ]:
!pip install -q psycopg2-binary
print("Driver installed.")

## Step 2 — Connect to your Neon database

Run the cell, then **paste your pooled connection string** at the prompt (input is hidden — it's a secret).

In [ ]:
import psycopg2
from getpass import getpass
from urllib.parse import urlparse

# getpass hides what you type. Prefer to see it? Swap getpass(...) for input(...).
NEON_DB_URL = getpass("Paste your Neon POOLED connection string: ").strip()

if not NEON_DB_URL.startswith(("postgres://", "postgresql://")):
    raise ValueError("That doesn't look like a Neon connection string — it should start with postgresql://")

conn = psycopg2.connect(NEON_DB_URL)
conn.autocommit = True   # every cell is self-contained, no dangling transactions
cur = conn.cursor()

cur.execute("SELECT version();")
print("[OK] Connected:", cur.fetchone()[0][:70] + "...")

host = (urlparse(NEON_DB_URL).hostname or "")
if "-pooler" in host:
    print("[OK] Pooled connection detected — exactly what the app wants.")
else:
    print("[!] Note: no '-pooler' in the host.")
    print("    Setup will still work, but when you deploy the app, use the POOLED string")
    print("    (Neon console -> Connect -> tick 'Pooled connection').")

## Step 3 — Create the tables

Creates `users` and `credentials` (idempotent — safe to re-run).

In [ ]:
DDL = """
-- Registered accounts. Passwords are stored as salted scrypt hashes.
CREATE TABLE IF NOT EXISTS users (
    id            TEXT PRIMARY KEY,
    username      TEXT NOT NULL UNIQUE,
    password_hash TEXT NOT NULL,
    created_at    TIMESTAMPTZ NOT NULL DEFAULT NOW(),
    updated_at    TIMESTAMPTZ NOT NULL DEFAULT NOW()
);

-- One row per stored 2FA credential.
-- secret_enc = AES-256-GCM encrypted setup key: base64(iv || tag || ciphertext)
CREATE TABLE IF NOT EXISTS credentials (
    id          TEXT PRIMARY KEY,
    user_id     TEXT NOT NULL REFERENCES users(id) ON DELETE CASCADE,
    label       TEXT NOT NULL,
    issuer      TEXT,
    secret_enc  TEXT NOT NULL,
    type        TEXT NOT NULL DEFAULT 'totp',        -- 'totp' | 'hotp'
    algorithm   TEXT NOT NULL DEFAULT 'SHA-1',       -- 'SHA-1' | 'SHA-256' | 'SHA-512'
    digits      INTEGER NOT NULL DEFAULT 6,
    period      INTEGER NOT NULL DEFAULT 30,         -- seconds (TOTP)
    counter     INTEGER NOT NULL DEFAULT 0,          -- HOTP sequence number
    encoding    TEXT NOT NULL DEFAULT 'base32',      -- 'base32' | 'steam'
    sort_order  BIGINT  NOT NULL DEFAULT 0,
    created_at  TIMESTAMPTZ NOT NULL DEFAULT NOW(),
    updated_at  TIMESTAMPTZ NOT NULL DEFAULT NOW()
);

-- The only index that matters: listing a user's credentials.
CREATE INDEX IF NOT EXISTS idx_credentials_user_id ON credentials (user_id);
"""

cur.execute(DDL)
print("[OK] Tables created (or already existed).")

cur.execute("""
    SELECT table_name FROM information_schema.tables
    WHERE table_schema = 'public' ORDER BY table_name;
""")
tables = [row[0] for row in cur.fetchall()]
print("Tables now in the database:", tables)

assert "users" in tables and "credentials" in tables, "Expected tables are missing!"
print("[OK] Schema verified: users + credentials are ready.")

## Step 4 — Generate your secret keys

The app needs two secrets set as Vercel environment variables. Run the cell, then **copy both values**.

In [ ]:
import base64
import secrets

ENCRYPTION_KEY = base64.b64encode(secrets.token_bytes(32)).decode("ascii")
SESSION_SECRET = secrets.token_urlsafe(36)

print("Copy these two values into Vercel -> Settings -> Environment Variables:")
print()
print("ENCRYPTION_KEY =", ENCRYPTION_KEY)
print("SESSION_SECRET =", SESSION_SECRET)
print()
print("(ENCRYPTION_KEY encrypts 2FA secrets at rest; SESSION_SECRET signs login cookies.)")
print("Store them somewhere safe — losing ENCRYPTION_KEY makes old secrets unreadable.")

## You're done with the database ✓

Next steps (see `SETUP.md` in the deployment zip):

1. Push the project to GitHub.
2. Import it on Vercel and add these environment variables:
   - `DATABASE_URL` = the pooled connection string you used above
   - `ENCRYPTION_KEY` and `SESSION_SECRET` = the values printed above
   - `NEXT_PUBLIC_SITE_URL` = `https://2fa.namansoni.in`
3. Deploy, then add the custom domain `2fa.namansoni.in` in Vercel.
4. Check `https://2fa.namansoni.in/api/health` — everything should say `true`.

---

### Optional utilities below — only run if you know why

#### Utility — database report
Shows table sizes and row counts. Handy for a quick health look.

In [ ]:
for table in ("users", "credentials"):
    cur.execute(f"SELECT COUNT(*) FROM {table};")
    count = cur.fetchone()[0]
    print(f"{table:<14} {count:>6} rows")

cur.execute("SELECT indexname FROM pg_indexes WHERE schemaname = 'public';")
print("\nIndexes:", [row[0] for row in cur.fetchall()])

#### Utility — full reset (DESTRUCTIVE)
Deletes every account and stored credential. There is no undo. Only run this if you truly want to wipe the platform's data (e.g. before a fresh launch).

In [ ]:
confirm = input("Type DROP (uppercase) to permanently delete ALL data: ")
if confirm.strip() == "DROP":
    cur.execute("DROP TABLE IF EXISTS credentials CASCADE;")
    cur.execute("DROP TABLE IF EXISTS users CASCADE;")
    print("[DONE] All data removed. Re-run Step 3 to recreate empty tables.")
else:
    print("Cancelled — nothing was deleted.")

In [ ]:
# Close the connection when you're finished with this notebook.
cur.close()
conn.close()
print("Connection closed. Setup complete.")